# TP N° 2 – MLP — Exercice 02

## Implémentation d'un MLP sur CIFAR-10

**Objectif :** Classifier les images couleur 32×32 de CIFAR-10 en 10 classes avec l’architecture imposée.

**Classes :** avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion.

### Spécifications rappelées
- **Données :** CIFAR-10 — 50 000 images d’entraînement, 10 000 de test, RGB 32×32 → vecteur **3072**.
- **Architecture :** Entrée 3072 → Dense 512 (ReLU) → Dense 256 (ReLU) → Sortie 10 (Softmax).
- **Compilation :** Adam, `categorical_crossentropy`, précision.
- **Entraînement :** 10 époques, batch 64, **validation_split=0.2** sur les données d’entraînement.

**Téléchargement :** `keras.datasets.cifar10.load_data()` utilise le serveur de Toronto, qui peut répondre **HTTP 503**. Le code ci-dessous **réessaie** quelques fois, puis bascule sur le jeu **`uoft-cs/cifar10`** sur Hugging Face (CDN). Si ce repli s’affiche, installe une fois : `pip install datasets`.

In [1]:
import time

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical


def load_cifar10_data():
    """Quelques essais Keras (503 Toronto souvent transitoire), puis Hugging Face (CDN)."""
    keras_exc = None
    for attempt in range(3):
        try:
            from tensorflow.keras.datasets import cifar10

            return cifar10.load_data()
        except Exception as exc:
            keras_exc = exc
            time.sleep(3)

    try:
        from datasets import load_dataset
    except ImportError as imp_exc:
        raise ImportError(
            "Le téléchargement Keras a échoué après 3 essais ; installez :\n"
            "  pip install datasets\n"
            f"Dernière erreur Keras : {keras_exc}"
        ) from imp_exc

    ds = load_dataset("uoft-cs/cifar10")
    train = ds["train"]
    test = ds["test"]
    img_col = "img" if "img" in train.column_names else "image"

    def split_to_xy(sub):
        x = np.stack([np.asarray(im, dtype=np.uint8) for im in sub[img_col]], axis=0)
        y = np.asarray(sub["label"], dtype=np.uint8).reshape(-1, 1)
        return x, y

    x_train, y_train = split_to_xy(train)
    x_test, y_test = split_to_xy(test)
    print("CIFAR-10 chargé via Hugging Face (uoft-cs/cifar10).")
    return (x_train, y_train), (x_test, y_test)


(x_train, y_train), (x_test, y_test) = load_cifar10_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

# Aplatissement (32, 32, 3) -> 3072 et normalisation [0, 1]
x_train = x_train.reshape(-1, 3072).astype("float32") / 255.0
x_test = x_test.reshape(-1, 3072).astype("float32") / 255.0

y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

model = Sequential(
    [
        Input(shape=(3072,)),
        Dense(512, activation="relu"),
        Dense(256, activation="relu"),
        Dense(10, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

history = model.fit(
    x_train,
    y_train_cat,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1,
)

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Perte (test) : {test_loss:.4f} — Précision (test) : {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     1,573,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,707,274 (6.51 MB)

 Trainable params: 1,707,274 (6.51 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 26ms/step - accuracy: 0.3163 - loss: 1.9030 - val_accuracy: 0.3730 - val_loss: 1.7543
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.3907 - loss: 1.7016 - val_accuracy: 0.3931 - val_loss: 1.6816
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4252 - loss: 1.6166 - val_accuracy: 0.4158 - val_loss: 1.6370
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4399 - loss: 1.5667 - val_accuracy: 0.4415 - val_loss: 1.5572
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.4588 - loss: 1.5168 - val_accuracy: 0.4558 - val_loss: 1.5267
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4713 - loss: 1.4835 - val_accuracy: 0.4582 - val_loss: 1.5175
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - accuracy: 0.4794 - loss: 1.4618 - val_accuracy: 0.4628 - val_loss: 1.4927
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4889 - loss: 1.4333 - 